In [1]:
# Imports
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

from katabatic.models.ctgan.models import CTGANModel

ROOT set to: C:\Users\Prabu\Downloads\Katabatic


In [2]:
# Preprocess data
dataset_path = ROOT / "raw_data" / "nursery.csv"
output_path = ROOT / "discretized_data" / "nursery.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path))

Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\nursery.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\nursery.csv


In [4]:
# Run pipeline

input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "nursery")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "nursery" / "ctgan")


ctgan_kwargs = dict(
    epochs=100,                     
    batch_size=256,                
    noise_dim=128,                  
    generator_hidden=(512, 512),    
    discriminator_hidden=(512, 512),
    lr=2e-4,
    betas=(0.5, 0.9),
    lambda_gp=10.0,
    use_gradient_penalty=True,      
    clip_value=0.01,
    n_critic=5,
    gumbel_tau=0.5,                
    seed=42,
    device="cpu",
    backend="torch",
)


debug_model = CTGANModel(**ctgan_kwargs)
print("CTGAN will train for:", debug_model.cfg["epochs"], "epochs")
print("Generator hidden:", debug_model.cfg["generator_hidden"])
print("Discriminator hidden:", debug_model.cfg["discriminator_hidden"])

pipeline = TrainTestSplitPipeline(
    model=lambda: CTGANModel(**ctgan_kwargs)
)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print(result)


CTGAN will train for: 100 epochs
Generator hidden: [512, 512]
Discriminator hidden: [512, 512]
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
0    0.333333
1    0.329186
3    0.312018
4    0.025270
2    0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
0    0.333333
1    0.329090
3    0.312114
4    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
[CTGAN] Synthetic data saved:
  X -> C:\Users\Prabu\Downloads\Katabatic\synthetic\nursery\ctgan\x_synth.csv
  y -> C:\Users\Prabu\Downloads\Katabatic\synthetic\nursery\ctgan\y_synth.csv


C:\Users\Prabu\AppData\Local\pypoetry\Cache\virtualenvs\katabatic-AJr3_ocI-py3.11\Lib\site-packages\xgboost\training.py:199: UserWarning: [14:50:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results\nursery\ctgan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.0255
F1 Score: 0.0013

MLP:
Accuracy: 0.0382
F1 Score: 0.0243

RF:
Accuracy: 0.0255
F1 Score: 0.0013

XGBoost:
Accuracy: 0.0255
F1 Score: 0.0013
Train test split pipeline executed successfully.
